# Εισαγωγή στην Μηχανική Προτροπών
Η μηχανική προτροπών είναι η διαδικασία σχεδιασμού και βελτιστοποίησης προτροπών για εργασίες επεξεργασίας φυσικής γλώσσας. Περιλαμβάνει την επιλογή των κατάλληλων προτροπών, τη ρύθμιση των παραμέτρων τους και την αξιολόγηση της απόδοσής τους. Η μηχανική προτροπών είναι κρίσιμη για την επίτευξη υψηλής ακρίβειας και απόδοσης στα μοντέλα NLP. Σε αυτήν την ενότητα, θα εξερευνήσουμε τα βασικά της μηχανικής προτροπών χρησιμοποιώντας τα μοντέλα OpenAI για εξερεύνηση.


### Άσκηση 1: Διαχωρισμός σε tokens
Εξερευνήστε τον διαχωρισμό σε tokens χρησιμοποιώντας το tiktoken, έναν γρήγορο tokenizer ανοιχτού κώδικα από την OpenAI
Δείτε το [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) για περισσότερα παραδείγματα.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Άσκηση 2: Επαλήθευση Ρυθμίσεων Κλειδιού Microsoft Foundry Models

> **Σημείωση:** Τα GitHub Models θα καταργηθούν στο τέλος Ιουλίου 2026. Αυτή η άσκηση χρησιμοποιεί αντ’ αυτού τα [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), που προσφέρουν τον ίδιο κατάλογο μοντέλων για δοκιμή δωρεάν και την εμπειρία Azure AI Inference SDK.

Εκτελέστε τον παρακάτω κώδικα για να επιβεβαιώσετε ότι το endpoint σας στο Microsoft Foundry Models έχει ρυθμιστεί σωστά. Ο κώδικας δοκιμάζει απλά ένα βασικό prompt και επαληθεύει το αποτέλεσμα. Η είσοδος `oh say can you see` θα πρέπει να ολοκληρωθεί περίπου με το `by the dawn's early light..`


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

model_name = "gpt-4o-mini"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### Άσκηση 3: Κατασκευές
Εξερευνήστε τι συμβαίνει όταν ζητάτε από το LLM να επιστρέψει συμπληρώσεις για μια προτροπή σχετικά με ένα θέμα που μπορεί να μην υπάρχει, ή για θέματα που μπορεί να μην γνωρίζει γιατί ήταν εκτός του προ-εκπαιδευμένου συνόλου δεδομένων του (πιο πρόσφατα). Δείτε πώς αλλάζει η απάντηση αν δοκιμάσετε διαφορετική προτροπή ή διαφορετικό μοντέλο.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### Άσκηση 4: Βασισμένη σε Οδηγίες  
Χρησιμοποιήστε τη μεταβλητή "text" για να ορίσετε το κύριο περιεχόμενο  
και τη μεταβλητή "prompt" για να δώσετε μια οδηγία σχετική με αυτό το κύριο περιεχόμενο.  

Εδώ ζητάμε από το μοντέλο να συνοψίσει το κείμενο για έναν μαθητή της Β΄ τάξης  


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### Άσκηση 5: Σύνθετο Prompt 
Δοκιμάστε ένα αίτημα που περιλαμβάνει μηνύματα συστήματος, χρήστη και βοηθού 
Το σύστημα ορίζει το πλαίσιο του βοηθού
Τα μηνύματα Χρήστη & Βοηθού παρέχουν συμφραζόμενα συνομιλίας πολλαπλών γύρων

Σημειώστε πώς η προσωπικότητα του βοηθού ορίζεται σε "σαρκαστική" στο πλαίσιο του συστήματος. 
Δοκιμάστε να χρησιμοποιήσετε ένα διαφορετικό πλαίσιο προσωπικότητας. Ή δοκιμάστε μια διαφορετική σειρά μηνυμάτων εισόδου/εξόδου


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### Άσκηση: Εξερευνήστε την Ένστικτό σας
Τα παραπάνω παραδείγματα σας δίνουν μοτίβα που μπορείτε να χρησιμοποιήσετε για να δημιουργήσετε νέες εντολές (απλές, πολύπλοκες, με οδηγίες κλπ.) - δοκιμάστε να δημιουργήσετε άλλες ασκήσεις για να εξερευνήσετε μερικές από τις άλλες ιδέες που έχουμε συζητήσει όπως παραδείγματα, υποδείξεις και άλλα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
